# Financial Metrics - Latest WSB Stocks
This notebook refreshes financial metrics for the latest WallStreetBets tickers using current Yahoo Finance data.
It now:
- loads the latest top WSB stocks,
- fetches current Yahoo Finance price history,
- computes rolling standard deviation, SMA, EWMA, RSI, and MACD,
- runs a linear regression / RMSE check for each selected ticker,
- saves refreshed CSVs and charts under `outputs/latest_wsb_analysis/`.

## Step 1 - Setup

In [1]:
from pathlib import Path
import json
import sys
from typing import cast
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
try:
    ROOT = Path(__file__).resolve().parent
except NameError:
    ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.wsb_latest.pipeline import (
    compute_price_features,
    evaluate_linear_regression,
    fetch_price_history,
    plot_symbol_dashboard,
    run_pipeline,
)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
print(f"Project root: {ROOT}")

Project root: /home/runner/work/Project2-Sentiment-analysis-WSbets/Project2-Sentiment-analysis-WSbets


## Step 2 - Notebook configuration

In [2]:
OUTPUT_DIR = ROOT / "outputs" / "latest_wsb_analysis"
CHARTS_DIR = OUTPUT_DIR / "charts"
PRICES_DIR = OUTPUT_DIR / "prices"
PRICE_PERIOD = "1y"
TOP_K = 4
import os
REFRESH_WSB_DATA = os.getenv("WSB_REFRESH_DATA", "0") != "0"
MANUAL_TICKERS = None  # Example: ["AMD", "TSLA"]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHARTS_DIR.mkdir(parents=True, exist_ok=True)
PRICES_DIR.mkdir(parents=True, exist_ok=True)

## Step 3 - Refresh or load the latest WSB ranking

In [3]:
summary_path = OUTPUT_DIR / "top10_wsb_stocks.csv"
if REFRESH_WSB_DATA or (not summary_path.exists()):
    pipeline_result = run_pipeline(top_n=10, per_feed=100, price_period=PRICE_PERIOD, output_dir=OUTPUT_DIR)
    print(json.dumps(pipeline_result, indent=2))
summary_df = cast(pd.DataFrame, pd.read_csv(str(summary_path)))
summary_df["latest_mention"] = pd.to_datetime(summary_df["latest_mention"], utc=True)
summary_df[["ticker", "mention_count", "post_count", "avg_sentiment", "latest_mention"]]

,ticker,mention_count,post_count,avg_sentiment,latest_mention
0,AMD,12,6,0.273800,2026-07-07 20:16:28+00:00
1,WEN,9,6,0.361500,2026-07-06 17:47:13+00:00
2,NVDA,6,5,0.119220,2026-07-06 16:30:27+00:00
3,ZIM,15,1,0.993700,2026-07-02 19:59:29+00:00
4,HBM,8,2,0.623100,2026-07-07 01:09:55+00:00
5,SPY,4,3,0.483933,2026-07-07 20:24:58+00:00
6,TSLA,4,3,0.111100,2026-07-07 14:33:29+00:00
7,RKLB,3,3,0.298300,2026-07-04 22:43:24+00:00
8,AVAV,5,2,0.256700,2026-07-06 00:55:38+00:00
9,HOOD,4,2,0.581350,2026-07-03 18:54:03+00:00


## Step 4 - Select tickers for financial analysis

In [4]:
if MANUAL_TICKERS:
    selected_tickers = [ticker.upper() for ticker in MANUAL_TICKERS]
else:
    selected_tickers = summary_df["ticker"].head(TOP_K).tolist()
print("Selected tickers:", selected_tickers)

Selected tickers: ['AMD', 'WEN', 'NVDA', 'ZIM']


## Step 5 - Fetch current price history and compute financial metrics

In [5]:
def safe_latest_value(in_df, column_name, caster=float):
    if in_df.empty or column_name not in in_df.columns:
        return None
    col_data = in_df[column_name].dropna()
    if col_data.empty:
        return None
    return caster(col_data.iloc[-1])


price_feature_frames = {}
regression_frames = {}
analysis_rows = []
for ticker in selected_tickers:
    history_df = fetch_price_history(ticker, PRICE_PERIOD)
    features_df = compute_price_features(history_df)
    rmse, regression_df = evaluate_linear_regression(features_df)
    features_path = PRICES_DIR / f"{ticker.lower()}_price_features.csv"
    regression_path = PRICES_DIR / f"{ticker.lower()}_regression_results.csv"
    dashboard_path = CHARTS_DIR / f"{ticker.lower()}_dashboard.png"
    features_df.to_csv(features_path)
    regression_df.to_csv(regression_path)
    plot_symbol_dashboard(ticker, features_df, regression_df, dashboard_path)
    price_feature_frames[ticker] = features_df
    regression_frames[ticker] = regression_df
analysis_rows.append({
        "ticker": ticker,
        "rows": int(len(features_df)),
        "last_close": safe_latest_value(features_df, "Close", float),
        "latest_rsi": safe_latest_value(features_df, "RSI", float),
        "latest_macd": safe_latest_value(features_df, "MACD", float),
        "return_21d": safe_latest_value(features_df, "21D Return", float),
        "rmse": rmse,
        "features_csv": str(features_path),
        "regression_csv": str(regression_path),
        "dashboard_png": str(dashboard_path),
    })
analysis_df = pd.DataFrame(analysis_rows).sort_values("ticker").reset_index(drop=True)
analysis_summary_path = OUTPUT_DIR / "financial_metrics_summary.csv"
analysis_df.to_csv(analysis_summary_path, index=False)
analysis_df

,ticker,rows,last_close,latest_rsi,latest_macd,return_21d,rmse,features_csv,regression_csv,dashboard_png
0,ZIM,251,23.92,40.06184,-0.191633,-0.060118,0.730749,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...


## Step 6 - Inspect the computed metrics for each selected ticker

In [6]:
for ticker in selected_tickers:
    print(f"\n===== {ticker} latest metrics =====")
    display_cols = [
        "Close",
        "Volume",
        "30-Day Rolling STD",
        "30-Day Rolling SMA",
        "30-Day Rolling EWMA",
        "RSI",
        "MACD",
        "5D Return",
        "21D Return",
    ]
    print(price_feature_frames[ticker][display_cols].tail(5).to_string())


===== AMD latest metrics =====
                 Close    Volume  30-Day Rolling STD  30-Day Rolling SMA  30-Day Rolling EWMA        RSI       MACD  5D Return  21D Return
Date                                                                                                                                      
2026-06-30  580.909973  34488500           39.191054          502.851333           393.508625  64.942421  27.586782   0.117457    0.125576
2026-07-01  540.880005  28326800           36.581512          506.847666           396.885557  56.477972  26.342939   0.040674    0.060279
2026-07-02  517.820007  28142500           32.140878          510.306667           399.656496  52.252854  23.228670  -0.027696   -0.007133
2026-07-06  552.049988  30684200           30.739234          513.789000           403.147993  57.352976  23.254598   0.058419    0.017566
2026-07-07  516.109985  29007561           28.246753          516.006333           405.735889  51.172449  20.142894  -0.043337   -0.01

## Step 7 - Plot the latest closing prices for the selected tickers

In [7]:
fig, ax = plt.subplots(figsize=(12, 6))
for ticker in selected_tickers:
    close_series = price_feature_frames[ticker]["Close"].dropna()
    ax.plot(close_series.index, close_series.values, label=ticker, linewidth=1.8)
ax.set_title("Latest selected WSB tickers - close prices")
ax.set_ylabel("Close")
ax.legend(loc="upper left")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Step 8 - Review the regression predictions

In [8]:
for ticker in selected_tickers:
    regression_df = regression_frames[ticker]
    print(f"\n===== {ticker} regression results =====")
    if regression_df.empty:
        print("Not enough data for regression output.")
    else:
        print(regression_df.tail(10).to_string())


===== AMD regression results =====
                 Close  Predicted Close
Date                                   
2026-06-23  519.849976       506.429790
2026-06-24  519.739990       509.121024
2026-06-25  532.570007       512.704205
2026-06-26  521.580017       507.991869
2026-06-29  539.489990       513.914375
2026-06-30  580.909973       535.260159
2026-07-01  540.880005       526.543126
2026-07-02  517.820007       517.844878
2026-07-06  552.049988       529.539682
2026-07-07  516.109985       517.815672

===== WEN regression results =====
            Close  Predicted Close
Date                              
2026-06-23   6.26         6.368915
2026-06-24   7.86         7.186292
2026-06-25   7.33         7.067103
2026-06-26   7.80         7.372692
2026-06-29   8.26         7.775313
2026-06-30   8.29         7.835326
2026-07-01   8.94         8.032808
2026-07-02   8.60         7.973224
2026-07-06   7.90         7.598210
2026-07-07   7.78         7.513602

===== NVDA regression resul

## Step 9 - Final notebook summary

In [9]:
notebook_summary = {
    "selected_tickers": selected_tickers,
    "price_period": PRICE_PERIOD,
    "top_k": TOP_K,
    "analysis_summary_csv": str(analysis_summary_path.resolve()),
    "output_dir": str(OUTPUT_DIR.resolve()),
}
print(json.dumps(notebook_summary, indent=2))

{
  "selected_tickers": [
    "AMD",
    "WEN",
    "NVDA",
    "ZIM"
  ],
  "price_period": "1y",
  "top_k": 4,
  "analysis_summary_csv": "/home/runner/work/Project2-Sentiment-analysis-WSbets/Project2-Sentiment-analysis-WSbets/outputs/latest_wsb_analysis/financial_metrics_summary.csv",
  "output_dir": "/home/runner/work/Project2-Sentiment-analysis-WSbets/Project2-Sentiment-analysis-WSbets/outputs/latest_wsb_analysis"
}
